<a href="https://colab.research.google.com/github/junseo25/ds2002-fa26/blob/main/notebooks/04-json-apis/2026_09_30_Jun_Seo_Lee_%E2%80%94_JSON_to_Tidy_Tables_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · JSON to Tidy Tables

**Studio — 2026-09-30 · Fall 2026**  
**Class time:** 45 minutes

---

## One response, straight off the wire

Monday you took apart a payload that had already been laid out for you, indented and readable. Real responses do not arrive that way. Whitespace costs bandwidth, so APIs ship their JSON with every space squeezed out of it.

Here is a portfolio response from a market-data service, exactly as it came down. Run the cell and try to read it.

In [1]:
RAW = '{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple Inc.","sector":"Technology"},"shares":120,"prices":[{"date":"2026-09-28","close":231.4,"volume":52300000},{"date":"2026-09-29","close":229.85,"volume":48100000},{"date":"2026-09-30","close":233.1,"volume":55600000}]},{"ticker":"MSFT","company":{"name":"Microsoft Corp.","sector":"Technology"},"shares":45,"prices":[{"date":"2026-09-28","close":502.1,"volume":22100000},{"date":"2026-09-29","close":"498.75","volume":0},{"date":"2026-09-30","close":505.3,"volume":24800000}]},{"ticker":"KO","company":{"name":"Coca-Cola Co."},"shares":300,"prices":[{"date":"2026-09-28","close":71.25,"volume":12400000},{"date":"2026-09-29","close":null,"volume":13000000},{"date":"2026-09-30","close":70.9,"volume":11800000}]}]}}'
print('characters:', len(RAW))
print(RAW[:160], '...')

characters: 871
{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple I ...


Eight hundred and seventy-one characters on one line. You cannot read that, and neither can anyone else. That is fine. **Nobody reads raw JSON.** You parse it, then you ask Python to lay it out for you.

Notice what you are holding right now: `RAW` is a *string*. It has a length. You could slice it. There are no keys to look up yet, because text does not have keys.

### Step 1 — parse it, then make it readable

Two different jobs, and people confuse them constantly. `json.loads` turns text into Python objects. `json.dumps` with `indent` turns Python objects back into text that a human can scan. You are about to do both, in that order, and end up looking at the same data you started with — just formatted.

In [2]:
import json, pandas as pd

data = json.loads(RAW)
print(type(RAW), '->', type(data))
print(json.dumps(data, indent=2)[:700], '...')

<class 'str'> -> <class 'dict'>
{
  "as_of": "2026-09-30",
  "currency": "USD",
  "source": "teaching-fixture",
  "portfolio": {
    "name": "Cavalier Growth",
    "holdings": [
      {
        "ticker": "AAPL",
        "company": {
          "name": "Apple Inc.",
          "sector": "Technology"
        },
        "shares": 120,
        "prices": [
          {
            "date": "2026-09-28",
            "close": 231.4,
            "volume": 52300000
          },
          {
            "date": "2026-09-29",
            "close": 229.85,
            "volume": 48100000
          },
          {
            "date": "2026-09-30",
            "close": 233.1,
            "volume": 55600000
          }
        ]
      },
      { ...


Same information, 871 characters versus something much longer, and now you can see the shape. `indent=2` is the first thing to reach for when a payload lands in your lap.

**TODO:** in one sentence, why is it wrong to say the API "sent you a dictionary"?

It is incorrect to say the API sent a dictionary because the API sent JSON text, which json.loads() parsed into a Python dictionary

### Step 2 — map the shape before you touch it

Never normalize a payload you have not walked. You need two numbers before you go any further: how many holdings there are, and how many price rows you should end up with.

The first line is done for you — `.keys()` gives you an object's names when you do not already know them. The counting is yours.

**TODO:** print the portfolio name, the number of holdings, and the total number of price records across all holdings.

In [3]:
print('top-level keys:', list(data.keys()))

# Walk the nested shape before flattening it.
portfolio_name = data['portfolio']['name']
holdings = data['portfolio']['holdings']
holdings_count = len(holdings)
price_rows = sum(len(holding['prices']) for holding in holdings)

print('portfolio:', portfolio_name)
print('holdings:', holdings_count)
print('price rows:', price_rows)

top-level keys: ['as_of', 'currency', 'source', 'portfolio']
portfolio: Cavalier Growth
holdings: 3
price rows: 9


**TODO:** write your two numbers down here, because every assertion below checks against them.

- holdings: 3
- price rows: 9
- one row of the table you are about to build is one ticker on one date

### Step 3 — trace a path, and hit the wall

A path is the sequence of names and positions that gets you to one value. Reading a path is the skill; the brackets are just typing.

**TODO:** print Apple's sector, then try to print Coca-Cola's the same way. One of those raises an exception. Catch it and read what it says. Then rewrite the line so it checks whether the name is there first and prints `'unknown'` when it is not — the `'pm25' not in r` test from Monday's missing/null/empty/zero table, pointed at `'sector'`.

In [4]:
# Apple has a sector field, so the complete path succeeds.
apple_sector = data['portfolio']['holdings'][0]['company']['sector']
print('apple sector:', apple_sector)

# Coca-Cola does not have that key; catching KeyError makes the missing field visible.
try:
    ko_sector = data['portfolio']['holdings'][2]['company']['sector']
    print('ko sector:', ko_sector)
except KeyError as err:
    print('error:', err)

# Check membership before lookup so a missing sector receives an honest label.
ko_company = data['portfolio']['holdings'][2]['company']
if 'sector' in ko_company:
    ko_sector = ko_company['sector']
else:
    ko_sector = 'unknown'
print('ko sector:', ko_sector)

apple sector: Technology
error: 'sector'
ko sector: unknown


### Step 4 — the four checks

It decoded and it parsed, so checks 1 and 2 passed the moment `json.loads` returned. Check 3 asks whether the data matches what you were promised, and check 4 asks whether it means what you think.

This payload has **four** planted problems. Three of them are in the price records and one is in a company object. None of them will raise an error.

**TODO:** walk every price record and report, with the ticker and date for each:

1. a `close` that came through as `None`
2. a `close` that is a *string* when every other one is a number
3. a `volume` of exactly `0`
4. a holding whose company object is missing a field the others have

Then answer the one that matters: **which of those four are genuinely missing data, and which are real measurements?** They do not get handled the same way.

In [5]:
# Inspect every nested record for missing values, type drift, and a real zero.
for holding in holdings:
    ticker = holding['ticker']
    company = holding['company']

    if 'sector' not in company:
        print(ticker, 'is missing company sector')

    for price_record in holding['prices']:
        date_value = price_record['date']
        close = price_record['close']
        volume = price_record['volume']

        if close is None:
            print(ticker, date_value, 'has a missing close')
        if isinstance(close, str):
            print(ticker, date_value, 'has close stored as text:', close)
        if volume == 0:
            print(ticker, date_value, 'has zero volume')

MSFT 2026-09-29 has close stored as text: 498.75
MSFT 2026-09-29 has zero volume
KO is missing company sector
KO 2026-09-29 has a missing close


**TODO:** your answer to the last question: Coca-Cola’s missing sector and its null closing price are genuinely missing data. Microsoft’s "498.75" closing price is present but stored using the wrong type, so it should be converted to numeric. Microsoft’s volume of 0 is a real recorded measurement rather than a missing value and should not automatically be removed.

### Step 5 — flatten to one row per ticker per day

`record_path` is the list you want one row from. `meta` is everything above it that should be copied down onto each of those rows. `sector` only exists on two of the three companies, so you will need `errors='ignore'` to keep `json_normalize` from refusing.

*Expected: 9 rows.*

In [6]:
# One output row represents one ticker on one date.
px = pd.json_normalize(
    holdings,
    record_path='prices',
    meta=['ticker', 'shares', ['company', 'name'], ['company', 'sector']],
    sep='.',
    errors='ignore',
)

assert len(px) == 9
print(px)
print('\ncolumn types:')
print(px.dtypes)

         date   close    volume ticker shares     company.name company.sector
0  2026-09-28   231.4  52300000   AAPL    120       Apple Inc.     Technology
1  2026-09-29  229.85  48100000   AAPL    120       Apple Inc.     Technology
2  2026-09-30   233.1  55600000   AAPL    120       Apple Inc.     Technology
3  2026-09-28   502.1  22100000   MSFT     45  Microsoft Corp.     Technology
4  2026-09-29  498.75         0   MSFT     45  Microsoft Corp.     Technology
5  2026-09-30   505.3  24800000   MSFT     45  Microsoft Corp.     Technology
6  2026-09-28   71.25  12400000     KO    300    Coca-Cola Co.            NaN
7  2026-09-29    None  13000000     KO    300    Coca-Cola Co.            NaN
8  2026-09-30    70.9  11800000     KO    300    Coca-Cola Co.            NaN

column types:
date                 str
close             object
volume             int64
ticker               str
shares            object
company.name         str
company.sector       str
dtype: object


### Step 6 — fix the type drift

One string in a column of numbers is enough to make the whole column text, and a text column will not average. `pd.to_numeric(..., errors='coerce')` converts what it can and turns the rest into `NaN`.

**TODO:** convert `close` to a number, then confirm the dtype changed and count how many nulls you are left with. Say out loud where that null came from — it is not the string.

In [7]:
# Numeric text converts successfully; the original null remains missing.
px['close'] = pd.to_numeric(px['close'], errors='coerce')
print('close dtype:', px['close'].dtype)
print('missing close values:', px['close'].isna().sum())

close dtype: float64
missing close values: 1


### Step 7 — answer the questions

**TODO:** produce each of these, and report `count` next to every average so a reader can see how much data it rests on.

1. Mean closing price per ticker, and the number of days each mean is built from.
2. Each position's value on **2026-09-30** — that is `shares × close` — plus the total portfolio value and which holding is the largest.
3. Mean daily volume across all nine rows. Then compute it again with the zero-volume day excluded and report the difference.

That third one is the real question of the day. **Was dropping the zero the right call?** Defend your answer in a sentence.

In [8]:
# Count is displayed beside each mean so the amount of supporting data is visible.
ticker_summary = (px.groupby('ticker')
                  .agg(mean_close=('close', 'mean'),
                       n_observed=('close', 'count'))
                  .round(2))
print(ticker_summary)

last_day = px.loc[px['date'] == '2026-09-30'].copy()
last_day['position_value'] = last_day['shares'] * last_day['close']
total_portfolio_value = last_day['position_value'].sum()
largest_position = last_day.loc[last_day['position_value'].idxmax(), 'ticker']

print('\nposition values:')
print(last_day[['ticker', 'shares', 'close', 'position_value']])
print(f'total portfolio value: ${total_portfolio_value:,.2f}')
print('largest position:', largest_position)

# Keep the zero because it was reported as a real measurement, not a missing value.
mean_volume_with_zero = px['volume'].mean()
mean_volume_without_zero = px.loc[px['volume'] != 0, 'volume'].mean()
difference = mean_volume_without_zero - mean_volume_with_zero
print(f'with zero: {mean_volume_with_zero:,.1f}')
print(f'without zero: {mean_volume_without_zero:,.1f}')
print(f'difference: {difference:,.1f}')

        mean_close  n_observed
ticker                        
AAPL        231.45           3
KO           71.08           2
MSFT        502.05           3

position values:
  ticker shares  close position_value
2   AAPL    120  233.1        27972.0
5   MSFT     45  505.3        22738.5
8     KO    300   70.9        21270.0
total portfolio value: $71,980.50
largest position: AAPL
with zero: 26,677,777.8
without zero: 30,012,500.0
difference: 3,334,722.2


**TODO:** the zero-volume question: I kept the zero-volume observation because it was explicitly reported as 0, not as null or a missing field. Excluding it would treat a real measurement as missing and increase the reported mean by approximately 3.3 million shares.

### Step 8 — write the result back out

Your output is somebody else's input, so send the context with it — the date and the currency, not a bare table of numbers.

**TODO:** write your per-ticker summary to `portfolio_summary.json` with `indent=2` and `allow_nan=False`, then print the file back. If `allow_nan=False` raises, that is the point of it — a `NaN` reached your output and you have to decide what it should say before it leaves your machine.

In [9]:
# Include source context with the summary so the output is meaningful on its own.
result = {
    'as_of': data['as_of'],
    'currency': data['currency'],
    'portfolio': data['portfolio']['name'],
    'summary': ticker_summary.reset_index().to_dict('records'),
}

with open('portfolio_summary.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, allow_nan=False)

with open('portfolio_summary.json', encoding='utf-8') as f:
    print(f.read())

{
  "as_of": "2026-09-30",
  "currency": "USD",
  "portfolio": "Cavalier Growth",
  "summary": [
    {
      "ticker": "AAPL",
      "mean_close": 231.45,
      "n_observed": 3
    },
    {
      "ticker": "KO",
      "mean_close": 71.08,
      "n_observed": 2
    },
    {
      "ticker": "MSFT",
      "mean_close": 502.05,
      "n_observed": 3
    }
  ]
}


---

## Part 2 — a second payload, if you have time

This one is already formatted, and it is deeper: a game-day response nested three levels down. Nothing here is new — it is the same `json_normalize` from Step 5 reaching further down, then the `merge` you have been using since Sep 14. The only twist is that you pull **two** tables out of one payload and join them back together.

In [10]:
response = {
    'game': 'UVA vs Duke',
    'date': '2026-10-10',
    'attendance': 47200,
    'zones': [
        {'zone': 'A', 'capacity': 400, 'vendors': [
            {'vendor_id': 'V-01', 'name': 'Hoos Burgers',
             'sales': {'orders': 300, 'revenue': 2250.0},
             'lines': [{'item': 'Cheeseburger', 'qty': 180},
                       {'item': 'Hot Dog', 'qty': 120}]},
            {'vendor_id': 'V-07', 'name': 'Stadium Dogs',
             'sales': {'orders': 210},
             'lines': [{'item': 'Hot Dog', 'qty': 210}]},
        ]},
        {'zone': 'B', 'capacity': 150, 'vendors': [
            {'vendor_id': 'V-12', 'name': 'Wahoo Wings',
             'sales': {'orders': 150, 'revenue': 1800.0},
             'lines': [{'item': 'Wings 6pc', 'qty': 90},
                       {'item': 'Wings 12pc', 'qty': 60}]},
        ]},
        {'zone': 'C', 'capacity': 120, 'vendors': [
            {'vendor_id': 'V-18', 'name': 'Rally Rain Gear',
             'sales': {'orders': 95, 'revenue': 570.0},
             'lines': [{'item': 'Rain Poncho', 'qty': 95}]},
        ]},
    ],
}
print('vendors:', sum(len(z['vendors']) for z in response['zones']))
print('line items:', sum(len(v['lines'])
                         for z in response['zones'] for v in z['vendors']))

vendors: 4
line items: 6


Four vendors, six line items.

**TODO A.** Flatten to one row per vendor, carrying down the game, the date, and the zone's `zone` and `capacity`. Use `sep='_'` and leave the column names it gives you alone. *4 rows.*

**TODO B.** One vendor is missing a field. Find it, and add a boolean column recording whether revenue was reported. Do **not** fill it with zero — write a comment saying why that would be wrong here.

**TODO C.** Flatten the same payload again, down to line items this time, keeping `vendor_id` and `zone`. *6 rows, quantities summing to 755.*

**TODO D.** Join the two, so each line item carries its vendor name and zone capacity. The vendor ID is not spelled the same way in both tables, so reach for `left_on` and `right_on` the way you did on Sep 16. Use `indicator=True` and confirm you still have 6 rows with nothing unmatched.

In [11]:
# Flatten the nested vendor list while carrying its parent fields down.
vendors = pd.json_normalize(
    response,
    record_path=['zones', 'vendors'],
    meta=['game', 'date', ['zones', 'zone'], ['zones', 'capacity']],
    sep='_',
    errors='ignore',
)
assert len(vendors) == 4
vendors

  vendor_id             name  ... zones_zone  zones_capacity
0      V-01     Hoos Burgers  ...          A             400
1      V-07     Stadium Dogs  ...          A             400
2      V-12      Wahoo Wings  ...          B             150
3      V-18  Rally Rain Gear  ...          C             120

[4 rows x 9 columns]

In [12]:
vendors['revenue_reported'] = (
    vendors['sales_revenue'].notna()
)

print(
    vendors[
        [
            'vendor_id',
            'name',
            'sales_revenue',
            'revenue_reported'
        ]
    ]
)

  vendor_id             name  sales_revenue  revenue_reported
0      V-01     Hoos Burgers         2250.0              True
1      V-07     Stadium Dogs            NaN             False
2      V-12      Wahoo Wings         1800.0              True
3      V-18  Rally Rain Gear          570.0              True


In [13]:
lines = pd.json_normalize(
    response,
    record_path=['zones', 'vendors', 'lines'],
    meta=[
        ['zones', 'vendors', 'vendor_id'],
        ['zones', 'zone']
    ],
    sep='_',
    errors='ignore'
)

assert len(lines) == 6
assert lines['qty'].sum() == 755

lines

           item  qty zones_vendors_vendor_id zones_zone
0  Cheeseburger  180                    V-01          A
1       Hot Dog  120                    V-01          A
2       Hot Dog  210                    V-07          A
3     Wings 6pc   90                    V-12          B
4    Wings 12pc   60                    V-12          B
5   Rain Poncho   95                    V-18          C

In [14]:
vendor_lines = lines.merge(
    vendors[
        [
            'vendor_id',
            'name',
            'zones_zone',
            'zones_capacity'
        ]
    ],
    left_on=[
        'zones_vendors_vendor_id',
        'zones_zone'
    ],
    right_on=[
        'vendor_id',
        'zones_zone'
    ],
    how='left',
    validate='many_to_one',
    indicator=True
)

print(vendor_lines['_merge'].value_counts())

assert len(vendor_lines) == 6
assert (
    vendor_lines['_merge'] == 'both'
).all()

vendor_lines

_merge
both          6
left_only     0
right_only    0
Name: count, dtype: int64


           item  qty  ... zones_capacity _merge
0  Cheeseburger  180  ...            400   both
1       Hot Dog  120  ...            400   both
2       Hot Dog  210  ...            400   both
3     Wings 6pc   90  ...            150   both
4    Wings 12pc   60  ...            150   both
5   Rain Poncho   95  ...            120   both

[6 rows x 8 columns]

---

## Checkpoint (participation)

Report your row counts from the portfolio payload and what you decided about the zero-volume day.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [15]:
# These checkpoint values are derived from the completed analysis above.
holdings_count = len(holdings)
price_rows = len(px)
largest_position = largest_position
null_vs_zero = 'KO close was missing; MSFT volume of 0 was a real measurement.'
zero_volume_call = 'I kept the zero because the source explicitly reported it.'

print('holdings:', holdings_count)
print('price rows:', price_rows)
print('largest position:', largest_position)
print('null vs zero:', null_vs_zero)
print('zero volume:', zero_volume_call)

holdings: 3
price rows: 9
largest position: AAPL
null vs zero: KO close was missing; MSFT volume of 0 was a real measurement.
zero volume: I kept the zero because the source explicitly reported it.
